# After storm area impact mapping using Sentinel-2 data

Map the extent of storm affected areas using Sentinel-2 data.

Generate a pre event mosaic and use the first available post event image. 
Calculate the NDVI for both and generate a change image using an NDVI threshold and the formula (pre-image * 10) + post-image to generate the following classes for each image:

- 00 indicates no change
- 01 change
- 10 is not reasonable because it means that pre-event is true and post not true and can be disregarded
- 11 both images are true --> in both images NDVI is below the threshold and no change happened

In [16]:
import openeo
import rasterio
from folium.plugins import Draw
import leafmap
from shapely.geometry import shape
from openeo.processes import quantiles, array_element, median
import time
from pathlib import Path
import numpy as np

In [5]:
# define properties for the outputs
out_dir = Path("/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/wind/vaia")
#out_dir.mkdir()
event = "vaia_2018"

## 1) Define time frame and extent

In [6]:
# dates 
PRE_DATE  = ("2018-08-01", "2018-09-30")   
POST_DATE = ("2019-06-28", "2019-06-28")   

In [7]:
# open a map and zoom to the area of interest
m = leafmap.Map(center=(46.65, 11.4), zoom=8.5)
m

Map(center=[46.65, 11.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

In [8]:
feat = m.draw_features
geom_dict = feat[0]['geometry']
geom = shape(geom_dict)

minx, miny, maxx, maxy = geom.bounds

bbox = {
    "west": minx,
    "south": miny,
    "east": maxx,
    "north": maxy,
}

print(bbox)

{'west': 11.476936, 'south': 46.373464, 'east': 11.571007, 'north': 46.424251}


### 2) Authentificate and load the pre and post Sentinel-2 cube

In [9]:
connection = openeo.connect("openeo.dataspace.copernicus.eu").authenticate_oidc()
connection.authenticate_oidc()

Authenticated using refresh token.
Authenticated using refresh token.


<Connection to 'https://openeo.dataspace.copernicus.eu/openeo/1.2/' with OidcBearerAuth>

In [11]:
BANDS   = ["B08", "B04"]         
BAD_SCL = [1, 3, 7, 8, 9, 10]

TIMESPAN = {"PRE": PRE_DATE, "POST": POST_DATE}   


def load_s2(temporal_extent):
    bands = connection.load_collection(
        "SENTINEL2_L2A",
        spatial_extent=bbox,
        temporal_extent=list(temporal_extent),
        bands=BANDS,
        max_cloud_cover=20,
    ).resample_spatial(resolution=10, method="bilinear")

    scl = connection.load_collection(
        "SENTINEL2_L2A",
        spatial_extent=bbox,
        temporal_extent=list(temporal_extent),
        bands=["SCL"],
        max_cloud_cover=50,
    ).resample_spatial(resolution=10, method="near")   # categorical -> nearest

    return bands, scl


def cloud_mask(scl):
    s = scl.band("SCL")
    m = s == BAD_SCL[0]
    for v in BAD_SCL[1:]:
        m = m | (s == v)
    return m


def masked_cube(temporal_extent):
    bands, scl = load_s2(temporal_extent)
    return bands.mask(cloud_mask(scl))


def composite_pre(temporal_extent):
    return masked_cube(temporal_extent).reduce_dimension(
        dimension="t",
        reducer=lambda x: array_element(quantiles(data=x, probabilities=[0.25]), index=0),
    )


def composite_post(temporal_extent):
    return masked_cube(temporal_extent).reduce_dimension(
        dimension="t",
        reducer=lambda x: median(x),
    )


def ndvi(mosaic):
    """NDVI = (NIR - Red) / (NIR + Red) = (B08 - B04) / (B08 + B04)"""
    def _ndvi(b):
        nir = b[0]   # B08
        red = b[1]   # B04
        return (nir - red) / (nir + red)
    return mosaic.reduce_dimension(dimension="bands", reducer=_ndvi)


COMPOSITES = {"PRE": composite_pre, "POST": composite_post}

# --- PRE and POST NDVI as batch jobs ---
jobs = {}
for name, extent in TIMESPAN.items():
    print(f"Starting {name}...")
    result = ndvi(COMPOSITES[name](extent)).save_result(format="GTiff")
    job = result.create_job(title=f"S2_NDVI_{name}")
    job.start()
    jobs[name] = job
    print(f"{name} started: {job.job_id}")
    time.sleep(10)

Starting PRE...
PRE started: j-26100709485544c5a83ed9c6a18e3f44
Starting POST...
POST started: j-2610070949084cffaf04e98abeedea51


In [12]:
file_pre_ndvi = out_dir/ f"pre_{event}_NDVI.tiff"
file_post_ndvi = out_dir/ f"post_{event}_NDVI.tiff"

jobs["PRE"].get_results().download_file(file_pre_ndvi)
jobs["POST"].get_results().download_file(file_post_ndvi)

PosixPath('/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/wind/vaia/post_vaia_2018_NDVI.tiff')

### 3) Calculate Change between pre and post image

In [17]:
thresh = 0.3

with rasterio.open(file_pre_ndvi) as src_pre:
    pre_ndvi = src_pre.read(1).astype("float32")
    profile = src_pre.profile
    pre_nodata = src_pre.nodata

with rasterio.open(file_post_ndvi) as src_post:
    post_ndvi = src_post.read(1).astype("float32")
    post_nodata = src_post.nodata

# Valid-pixel mask: NaN or declared nodata must not be classified
valid = ~np.isnan(pre_ndvi) & ~np.isnan(post_ndvi)
if pre_nodata is not None:
    valid &= pre_ndvi != pre_nodata
if post_nodata is not None:
    valid &= post_ndvi != post_nodata

pre_low  = (pre_ndvi  <= thresh).astype("uint8")
post_low = (post_ndvi <= thresh).astype("uint8")

change_ndvi = (pre_low * 10) + post_low
change_ndvi[~valid] = 255  # nodata

file_change_ndvi = out_dir / f"change_{event}_NDVI.tif"

profile.update(dtype="uint8", count=1, nodata=255, compress="lzw")
with rasterio.open(file_change_ndvi, "w", **profile) as dst:
    dst.write(change_ndvi, 1)